In [13]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [14]:
# ===== 1. VERİ ÜRETİMİ =====
np.random.seed(42)
X = np.linspace(0, 10, 200).reshape(-1, 1)
y = 3 * X.squeeze()**2 - 2 * X.squeeze() + 1 + np.random.randn(200) * 2
# Gerçek ilişki: y = 3x² - 2x + 1 + gürültü

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===== 2. BIAS-VARIANCE HESABI FONKSİYONU =====
def bias_variance(degree, X_train, y_train, X_test, y_test, n_iterations=100):

    predictions = []

    for _ in range(n_iterations):
        # Her iterasyonda farklı bootstrap örneği al
        indices = np.random.choice(len(X_train), len(X_train), replace=True)
        X_boot = X_train[indices]
        y_boot = y_train[indices]

        # Polinom özellikler oluştur ve modeli eğit
        poly = PolynomialFeatures(degree=degree)
        X_boot_poly = poly.fit_transform(X_boot)
        X_test_poly  = poly.transform(X_test)

        model = LinearRegression()
        model.fit(X_boot_poly, y_boot)

        predictions.append(model.predict(X_test_poly))

    predictions = np.array(predictions)  # (n_iterations, n_test)

    # Bias² = (ortalama tahmin - gerçek değer)²
    mean_pred = predictions.mean(axis=0)
    bias_sq   = np.mean((mean_pred - y_test) ** 2)

    # Variance = tahminlerin kendi aralarındaki değişkenliği
    variance  = np.mean(predictions.var(axis=0))

    # MSE = Bias² + Variance + Gürültü
    mse = mean_squared_error(y_test, mean_pred)

    return bias_sq, variance, mse

In [16]:
print(f"{'Derece':<10} {'Bias²':<12} {'Variance':<12} {'MSE':<12} {'Durum'}")
print("-" * 55)

for degree in [1, 2, 5, 10, 15]:
    bias_sq, variance, mse = bias_variance(degree, X_train, y_train, X_test, y_test)

    if bias_sq > 50:
        durum = "Underfitting (Yüksek Bias)"
    elif variance > 0.5:
        durum = "Overfitting (Yüksek Variance)"
    else:
        durum = "Dengeli ✅"

    print(f"{degree:<10} {bias_sq:<12.2f} {variance:<12.2f} {mse:<12.2f} {durum}")


Derece     Bias²        Variance     MSE          Durum
-------------------------------------------------------
1          386.59       6.33         386.59       Underfitting (Yüksek Bias)
2          3.09         0.05         3.09         Dengeli ✅
5          3.04         0.12         3.04         Dengeli ✅
10         3.18         0.23         3.18         Dengeli ✅
15         3.94         0.92         3.94         Overfitting (Yüksek Variance)
